# T3 — Real-ESRGAN spike (Colab free T4)
**EN:** Measures install recipe, weights hashes, seconds/frame, peak VRAM and encoder speed for a
decode -> enhance -> encode streaming pipeline. Set *Runtime -> Change runtime type -> T4 GPU*,
then *Runtime -> Run all* (about 15-25 min). Copy the JSON printed by the **last cell** back to the team.
Run it by hand once; there is no keep-alive or polling loop (Colab policy).

**TH:** สเปกนี้วัดสูตรติดตั้ง, SHA-256 ของ weights, วินาที/เฟรม, VRAM สูงสุด และความเร็ว encoder
ตั้ง Runtime เป็น T4 GPU แล้วกด Run all จากนั้นคัดลอก JSON จากเซลล์สุดท้ายส่งกลับ

## 1. GPU / environment check — ตรวจ GPU และสภาพแวดล้อม
**EN:** Fails fast if no GPU. Records versions, ffmpeg NVENC support, free disk.
**TH:** หยุดทันทีถ้าไม่มี GPU; บันทึกเวอร์ชัน, NVENC, พื้นที่ดิสก์

In [ ]:
import gc
import hashlib
import importlib
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import time
from fractions import Fraction
from pathlib import Path

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
WORK = ROOT / "spike"
WORK.mkdir(parents=True, exist_ok=True)

RESULTS = {
    "spike": "realesrgan",
    "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "env": {}, "install": {}, "weights": {}, "clips": {},
    "vram_probe": [], "jobs": [], "encoder_bench": [], "anime_smoke": {},
    "verify": [], "all_verify_pass": None,
}


def sh(cmd, check=True, timeout=None, env=None):
    """Run a command (list, or str via shell); return CompletedProcess with text output."""
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True,
                       timeout=timeout, env=env)
    if check and p.returncode != 0:
        tail = (p.stdout + p.stderr)[-3000:]
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}\n{tail}")
    return p


nvsmi = sh(["nvidia-smi"], check=False)
print(nvsmi.stdout or nvsmi.stderr)
if nvsmi.returncode != 0:
    raise RuntimeError("No GPU found. Runtime -> Change runtime type -> T4 GPU, then re-run.")

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError("torch.cuda.is_available() is False - select a GPU runtime.")

props = torch.cuda.get_device_properties(0)
smi_q = sh(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader"]).stdout.strip()
ffmpeg_ver = sh(["ffmpeg", "-version"]).stdout.splitlines()[0]
ffprobe_ver = sh(["ffprobe", "-version"]).stdout.splitlines()[0]
encoders = sh(["ffmpeg", "-hide_banner", "-encoders"]).stdout
nvenc_lines = [ln.strip() for ln in encoders.splitlines() if "nvenc" in ln]
disk = shutil.disk_usage(str(ROOT))

# ffmpeg >= 5.1 has -fps_mode; older builds (Colab Ubuntu 22.04 = 4.4) need -vsync 0. Feature-test it.
_t = sh(["ffmpeg", "-v", "error", "-f", "lavfi", "-i", "color=c=black:s=64x64:r=5", "-frames:v", "1",
         "-fps_mode", "passthrough", "-f", "null", "-"], check=False)
PASSTHROUGH = ["-fps_mode", "passthrough"] if _t.returncode == 0 else ["-vsync", "0"]


def nvenc_usable():
    """h264_nvenc listed AND able to encode 10 frames on this machine?"""
    if not any("h264_nvenc" in ln for ln in nvenc_lines):
        return False, "h264_nvenc not compiled into this ffmpeg"
    r = sh(["ffmpeg", "-v", "error", "-f", "lavfi", "-i", "testsrc2=size=1280x720:rate=30",
            "-frames:v", "10", "-c:v", "h264_nvenc", "-pix_fmt", "yuv420p", "-f", "null", "-"],
           check=False)
    return r.returncode == 0, (r.stderr or "").strip()[-300:]


NVENC_USABLE, NVENC_NOTE = nvenc_usable()

RESULTS["env"] = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu_name": props.name,
    "gpu_total_mib": props.total_memory // 2**20,
    "gpu_capability": f"{props.major}.{props.minor}",
    "nvidia_smi_query": smi_q,
    "ffmpeg": ffmpeg_ver,
    "ffprobe": ffprobe_ver,
    "ffmpeg_passthrough_flags": PASSTHROUGH,
    "nvenc_encoders_listed": nvenc_lines,
    "nvenc_usable": NVENC_USABLE,
    "nvenc_note": NVENC_NOTE,
    "disk_free_gib": round(disk.free / 2**30, 1),
}
print(json.dumps(RESULTS["env"], indent=2))

## 2. Install Real-ESRGAN (pinned) — ติดตั้ง Real-ESRGAN แบบ pin commit
**EN:** Clones `xinntao/Real-ESRGAN` at commit `a4abfb2979a7bbff3f69f58f58ae324608821e27` and puts it on `sys.path`
(no `pip install` of the repo itself, so its `setup.py` can't break). Installs `basicsr==1.4.2` with
`--no-deps` so Colab's torch/numpy are **not** touched; if the sdist fails to build (known
`KeyError: '__version__'` on Python 3.13+), it is downloaded, `setup.py` patched, and installed.
**TH:** ไม่ลดเวอร์ชัน torch ของ Colab; ติดตั้ง basicsr แบบ `--no-deps`

In [ ]:
import requests

REPO_URL = "https://github.com/xinntao/Real-ESRGAN"
REPO_COMMIT = "a4abfb2979a7bbff3f69f58f58ae324608821e27"
BASICSR_VERSION = "1.4.2"
REPO_DIR = WORK / "Real-ESRGAN"


def git(*args, check=True):
    return sh(["git", "-C", str(REPO_DIR), *args], check=check)


def pip(*args, check=True):
    return sh([sys.executable, "-m", "pip", "install", "-q", *args], check=check)


# --- 1) clone at the pinned commit -------------------------------------------------
if not (REPO_DIR / ".git").exists():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    sh(["git", "init", "-q", str(REPO_DIR)])
    git("remote", "add", "origin", REPO_URL)
    if git("fetch", "--depth", "1", "origin", REPO_COMMIT, check=False).returncode != 0:
        git("fetch", "origin")  # server refused fetch-by-sha: fall back to full fetch
git("checkout", "-q", "--force", REPO_COMMIT)
head = git("rev-parse", "HEAD").stdout.strip()
assert head == REPO_COMMIT, f"checked out {head} != pinned {REPO_COMMIT}"

# realesrgan/__init__.py does `from .version import *`; normally generated by setup.py.
ver_file = REPO_DIR / "VERSION"
repo_version = ver_file.read_text().strip() if ver_file.exists() else "0.0.0"
(REPO_DIR / "realesrgan" / "version.py").write_text(
    f"__version__ = {repo_version!r}\n__gitsha__ = {REPO_COMMIT[:7]!r}\n"
    f"version_info = tuple(int(x) for x in {repo_version!r}.split('.') if x.isdigit())\n")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# --- 2) light dependencies, never touching torch/torchvision/numpy -----------------------
pip("addict", "future", "lmdb", "pyyaml", "scikit-image", "scipy", "tqdm", "yapf", "requests")


# --- 3) basicsr (no deps) with fallback for the broken sdist build ----------------------
def install_basicsr_patched_sdist():
    meta = requests.get(f"https://pypi.org/pypi/basicsr/{BASICSR_VERSION}/json", timeout=60).json()
    sdist = next(u for u in meta["urls"] if u["packagetype"] == "sdist")
    dl_dir = WORK / "basicsr_sdist"
    dl_dir.mkdir(exist_ok=True)
    archive = dl_dir / sdist["filename"]
    archive.write_bytes(requests.get(sdist["url"], timeout=120).content)
    got = hashlib.sha256(archive.read_bytes()).hexdigest()
    assert got == sdist["digests"]["sha256"], "basicsr sdist hash mismatch"
    with tarfile.open(archive) as tf:
        if hasattr(tarfile, "data_filter"):
            tf.extractall(dl_dir, filter="data")
        else:
            tf.extractall(dl_dir)
    src_dir = next(p for p in dl_dir.iterdir() if p.is_dir() and (p / "setup.py").exists())
    setup_py = src_dir / "setup.py"
    text = setup_py.read_text()
    needle = "return locals()['__version__']"
    assert needle in text, "setup.py layout changed; patch needs updating"
    setup_py.write_text(text.replace(needle, f"return '{BASICSR_VERSION}'"))
    pip("--no-deps", str(src_dir))
    return {"sdist_sha256": got, "patched": "setup.py get_version() -> constant"}


basicsr_info = {}
if importlib.util.find_spec("basicsr") is None:
    r = pip("--no-deps", f"basicsr=={BASICSR_VERSION}", check=False)
    if r.returncode == 0:
        basicsr_info = {"method": "pip --no-deps"}
    else:
        print("pip install basicsr failed (expected on new Python) -> using patched sdist fallback. Tail:\n", (r.stdout + r.stderr)[-800:])
        basicsr_info = {"method": "patched sdist", **install_basicsr_patched_sdist()}
else:
    basicsr_info = {"method": "already installed"}
importlib.invalidate_caches()

RESULTS["install"] = {
    "repo": REPO_URL, "commit": head, "repo_version": repo_version,
    "basicsr_requested": BASICSR_VERSION, "basicsr": basicsr_info,
}
print(json.dumps(RESULTS["install"], indent=2))

## 3. Patch basicsr for new torchvision + import check — แพตช์ basicsr และทดสอบ import
**EN:** `basicsr` imports `torchvision.transforms.functional_tensor`, removed in torchvision 0.17+.
We rewrite that import in the installed package to `torchvision.transforms.functional`
(`rgb_to_grayscale` lives there) and print every changed line. Idempotent. The import test runs in a
fresh interpreter so no stale modules hide a failure.
**TH:** แก้ import ที่ถูกลบออกจาก torchvision ใหม่ แล้วทดสอบ import ในโปรเซสใหม่

In [ ]:
OLD_MOD = "torchvision.transforms.functional_tensor"
NEW_MOD = "torchvision.transforms.functional"


def module_exists(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False


spec = importlib.util.find_spec("basicsr")
assert spec and spec.submodule_search_locations, "basicsr not installed"
basicsr_dir = Path(list(spec.submodule_search_locations)[0])

legacy_module_present = module_exists(OLD_MOD)
patched = []
for py in sorted(basicsr_dir.rglob("*.py")):
    text = py.read_text(encoding="utf-8")
    if OLD_MOD in text:
        py.write_text(text.replace(OLD_MOD, NEW_MOD), encoding="utf-8")
        patched.append({
            "file": str(py.relative_to(basicsr_dir)),
            "before": [ln.strip() for ln in text.splitlines() if OLD_MOD in ln],
            "after": [ln.strip().replace(OLD_MOD, NEW_MOD) for ln in text.splitlines() if OLD_MOD in ln],
        })

print(f"legacy {OLD_MOD} importable: {legacy_module_present}")
print(f"basicsr dir: {basicsr_dir}")
print(f"files patched this run: {len(patched)}")
for item in patched:
    print(" ", item["file"])
    for b, a in zip(item["before"], item["after"]):
        print("     -", b)
        print("     +", a)
if not patched:
    print("(nothing to patch: already patched, or this basicsr no longer uses it)")

# Import check in a fresh interpreter.
env = dict(os.environ, PYTHONPATH=os.pathsep.join([str(REPO_DIR), os.environ.get("PYTHONPATH", "")]))
check_code = (
    "import basicsr, realesrgan;"
    "from basicsr.archs.rrdbnet_arch import RRDBNet;"
    "from realesrgan import RealESRGANer;"
    "from realesrgan.archs.srvgg_arch import SRVGGNetCompact;"
    "import numpy, cv2;"
    "print('basicsr', basicsr.__version__, '| realesrgan', realesrgan.version.__version__, "
    "'| numpy', numpy.__version__, '| cv2', cv2.__version__)"
)
chk = sh([sys.executable, "-c", check_code], check=False, env=env)
print(chk.stdout.strip())
if chk.returncode != 0:
    print(chk.stderr[-3000:])
    raise RuntimeError("Import check failed - see traceback above and send it back.")

RESULTS["install"]["torchvision_patch"] = {
    "legacy_module_present": legacy_module_present,
    "patched_files": patched,
    "import_check": chk.stdout.strip(),
}

## 4. Download weights + SHA-256 — ดาวน์โหลด weights และคำนวณ SHA-256
**EN:** Official GitHub release assets. Copy the two hashes into the results doc; they become the pinned values.
**TH:** ใช้ไฟล์จาก GitHub release ทางการ; เก็บ hash ไปล็อกในสเปกภายหลัง

In [ ]:
WEIGHT_URLS = {
    "RealESRGAN_x4plus.pth":
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    "realesr-animevideov3.pth":
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth",
}
WEIGHTS_DIR = WORK / "weights"
WEIGHTS_DIR.mkdir(exist_ok=True)
WEIGHTS = {}


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


for name, url in WEIGHT_URLS.items():
    dest = WEIGHTS_DIR / name
    t0 = time.perf_counter()
    if not dest.exists() or dest.stat().st_size == 0:
        tmp = dest.with_suffix(".part")
        with requests.get(url, stream=True, timeout=60, allow_redirects=True) as resp:
            resp.raise_for_status()
            with open(tmp, "wb") as f:
                for block in resp.iter_content(chunk_size=1 << 20):
                    f.write(block)
        tmp.replace(dest)
    WEIGHTS[name] = dest
    RESULTS["weights"][name] = {
        "url": url,
        "bytes": dest.stat().st_size,
        "sha256": sha256_file(dest),
        "download_sec": round(time.perf_counter() - t0, 2),
    }
    print(f"{name}\n  bytes  {RESULTS['weights'][name]['bytes']}\n  sha256 {RESULTS['weights'][name]['sha256']}")

## 5. Pipeline helpers — ฟังก์ชันช่วย (ffprobe / streaming / verify)
**EN:** Streaming design: ffmpeg decodes to raw RGB on a pipe, Python enhances one frame at a time,
a second ffmpeg encodes from stdin (video only), then a third ffmpeg muxes the original audio with
`-map 1:a? -c:a copy` (no `-shortest`). Exact rational FPS is read from ffprobe and passed to the encoder.
ffmpeg stderr goes to log files (never to a PIPE) so a full pipe can't deadlock.
**TH:** decode -> enhance ทีละเฟรม -> encode ผ่าน pipe แล้ว mux เสียงต้นฉบับ ไม่ใช้ `-shortest`

In [ ]:
import numpy as np

OOM_HINT = "out of memory"


def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or OOM_HINT in str(exc).lower()


def cuda_sync():
    torch.cuda.synchronize()


def probe(path):
    """Container/stream facts from ffprobe (no frame counting)."""
    out = sh(["ffprobe", "-v", "error", "-print_format", "json", "-show_streams",
              "-show_format", str(path)]).stdout
    d = json.loads(out)
    v = next(s for s in d["streams"] if s["codec_type"] == "video")
    audio = [s for s in d["streams"] if s["codec_type"] == "audio"]
    rate = v.get("r_frame_rate")
    if not rate or rate == "0/0":
        rate = v["avg_frame_rate"]
    return {
        "width": int(v["width"]), "height": int(v["height"]),
        "fps": rate,                        # exact rational string, e.g. "30000/1001"
        "avg_fps": v.get("avg_frame_rate"),
        "codec": v["codec_name"], "pix_fmt": v.get("pix_fmt"),
        "video_duration": float(v["duration"]) if "duration" in v else None,
        "format_duration": float(d["format"]["duration"]),
        "has_audio": bool(audio),
        "audio_codec": audio[0]["codec_name"] if audio else None,
        "audio_duration": float(audio[0]["duration"]) if audio and "duration" in audio[0] else None,
        "size_bytes": int(d["format"]["size"]),
    }


def count_frames(path):
    """Decode-count of video frames (slow but exact)."""
    out = sh(["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_frames",
              "-show_entries", "stream=nb_read_frames", "-of", "default=nokey=1:noprint_wrappers=1",
              str(path)]).stdout.strip()
    return int(out)


def encoder_args(name):
    if name == "libx264":
        return ["-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p"]
    if name == "h264_nvenc":
        return ["-c:v", "h264_nvenc", "-preset", "p5", "-rc", "vbr", "-cq", "18", "-b:v", "0",
                "-pix_fmt", "yuv420p"]
    raise ValueError(name)


def decode_cmd(src, extra=()):
    return ["ffmpeg", "-v", "error", "-i", str(src), "-map", "0:v:0", *PASSTHROUGH, *extra,
            "-f", "rawvideo", "-pix_fmt", "rgb24", "-"]


def encode_cmd(dst, width, height, fps, encoder):
    return ["ffmpeg", "-v", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
            "-s", f"{width}x{height}", "-framerate", fps, "-i", "-",
            *encoder_args(encoder), "-an", str(dst)]


def mux_cmd(video_only, original, dst):
    # Audio copied from the original; NO -shortest. '1:a?' = optional (clips without audio are fine).
    return ["ffmpeg", "-v", "error", "-y", "-i", str(video_only), "-i", str(original),
            "-map", "0:v:0", "-map", "1:a?", "-c:v", "copy", "-c:a", "copy",
            "-movflags", "+faststart", str(dst)]


def read_frames_bgr(src, n):
    """First n frames of src as BGR uint8 arrays (what RealESRGANer expects)."""
    info = probe(src)
    w, h = info["width"], info["height"]
    raw = subprocess.run(decode_cmd(src, ["-frames:v", str(n)]), capture_output=True, check=True).stdout
    fb = w * h * 3
    frames = [np.frombuffer(raw[i * fb:(i + 1) * fb], np.uint8).reshape(h, w, 3)[:, :, ::-1]
              for i in range(len(raw) // fb)]
    return [np.ascontiguousarray(f) for f in frames]


def build_upsampler(kind, tile):
    """kind: 'x4plus' (RRDBNet) or 'anime' (SRVGGNetCompact). fp16 on CUDA."""
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan import RealESRGANer
    from realesrgan.archs.srvgg_arch import SRVGGNetCompact

    if kind == "x4plus":
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        path = WEIGHTS["RealESRGAN_x4plus.pth"]
    elif kind == "anime":
        model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4,
                                act_type="prelu")
        path = WEIGHTS["realesr-animevideov3.pth"]
    else:
        raise ValueError(kind)
    up = RealESRGANer(scale=4, model_path=str(path), model=model, tile=tile, tile_pad=10,
                      pre_pad=0, half=True, device=torch.device("cuda"))
    # RealESRGANer's tiled path catches RuntimeError (incl. CUDA OOM), prints it and carries on,
    # leaving a black tile. Record such errors so we can fail loudly instead.
    up._swallowed = []
    orig_forward = up.model.forward

    def forward(*a, **k):
        try:
            return orig_forward(*a, **k)
        except RuntimeError as e:
            up._swallowed.append(str(e)[:200])
            raise

    up.model.forward = forward
    return up


def enhance_frame(up, bgr, outscale):
    up._swallowed.clear()
    with torch.no_grad():
        out, _ = up.enhance(bgr, outscale=outscale)
    if up._swallowed:
        raise RuntimeError("model error swallowed by RealESRGANer: " + up._swallowed[0])
    return out


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def enhance_video(src, dst, up, outscale, encoder, tile, label, log_every=30):
    """Streaming decode -> enhance -> encode -> mux. Returns a stats dict (never raises on OOM)."""
    src, dst = Path(src), Path(dst)
    info = probe(src)
    w, h, fps = info["width"], info["height"], info["fps"]
    video_only = WORK / f"{label}.video_only.mp4"
    up.tile_size = tile
    stats = {"label": label, "src": src.name, "dst": dst.name, "scale_arg": outscale,
             "encoder": encoder, "tile": tile, "in_size": [w, h], "fps": fps,
             "frames": 0, "ok": False, "error": None}
    t_read = t_model = t_write = 0.0
    dec = enc = None
    dec_log = open(WORK / f"{label}.decode.log", "wb")
    enc_log = open(WORK / f"{label}.encode.log", "wb")
    free_gpu()
    torch.cuda.reset_peak_memory_stats()
    t_start = time.perf_counter()
    try:
        dec = subprocess.Popen(decode_cmd(src), stdout=subprocess.PIPE, stderr=dec_log)
        frame_bytes = w * h * 3
        while True:
            t0 = time.perf_counter()
            buf = dec.stdout.read(frame_bytes)
            t1 = time.perf_counter()
            t_read += t1 - t0
            if len(buf) < frame_bytes:
                if buf:
                    raise RuntimeError(f"truncated frame ({len(buf)} of {frame_bytes} bytes)")
                break
            bgr = np.ascontiguousarray(np.frombuffer(buf, np.uint8).reshape(h, w, 3)[:, :, ::-1])
            out_bgr = enhance_frame(up, bgr, outscale)
            cuda_sync()
            t2 = time.perf_counter()
            t_model += t2 - t1
            if enc is None:
                oh, ow = out_bgr.shape[:2]
                if oh % 2 or ow % 2:
                    raise RuntimeError(f"odd output size {ow}x{oh} cannot be yuv420p")
                stats["out_size"] = [ow, oh]
                enc = subprocess.Popen(encode_cmd(video_only, ow, oh, fps, encoder),
                                       stdin=subprocess.PIPE, stderr=enc_log)
            enc.stdin.write(np.ascontiguousarray(out_bgr[:, :, ::-1]).data)
            t_write += time.perf_counter() - t2
            stats["frames"] += 1
            if stats["frames"] % log_every == 0:
                print(f"  [{label}] {stats['frames']} frames, "
                      f"{(time.perf_counter() - t_start) / stats['frames']:.3f} s/frame")
        t3 = time.perf_counter()
        if enc is not None:
            enc.stdin.close()
            enc.wait()
        stats["t_flush_sec"] = round(time.perf_counter() - t3, 3)
        if dec.wait() != 0 or (enc is not None and enc.returncode != 0):
            raise RuntimeError(f"ffmpeg failed: decode rc={dec.returncode}, "
                               f"encode rc={enc.returncode if enc else None}")
        t4 = time.perf_counter()
        sh(mux_cmd(video_only, src, dst))
        stats["t_mux_sec"] = round(time.perf_counter() - t4, 3)
        stats["ok"] = True
    except Exception as e:  # noqa: BLE001 - we want OOM and ffmpeg errors recorded, not raised
        stats["error"] = f"{type(e).__name__}: {str(e)[:300]}"
        stats["oom"] = is_oom(e)
    finally:
        for p in (dec, enc):
            if p is not None and p.poll() is None:
                p.kill()
                p.wait()
        dec_log.close()
        enc_log.close()
        video_only.unlink(missing_ok=True)
    total = time.perf_counter() - t_start
    n = max(stats["frames"], 1)
    stats.update({
        "t_total_sec": round(total, 3),
        "t_decode_read_sec": round(t_read, 3),
        "t_model_sec": round(t_model, 3),
        "t_encode_write_sec": round(t_write, 3),
        "sec_per_frame_total": round(total / n, 4),
        "sec_per_frame_model": round(t_model / n, 4),
        "peak_alloc_mib": round(torch.cuda.max_memory_allocated() / 2**20, 1),
        "peak_reserved_mib": round(torch.cuda.max_memory_reserved() / 2**20, 1),
    })
    free_gpu()
    return stats


def verify_output(src, dst, scale):
    """Compare output with input: frames, duration, fps, audio, format, size."""
    s, o = probe(src), probe(dst)
    sf, of = count_frames(src), count_frames(dst)
    dur_diff = abs(o["format_duration"] - s["format_duration"])
    checks = {
        "frame_count_equal": sf == of,
        "duration_diff_ok": dur_diff <= 0.1,
        "fps_equal": Fraction(o["fps"]) == Fraction(s["fps"]),
        "audio_preserved": (o["has_audio"] and o["audio_codec"] == s["audio_codec"]) if s["has_audio"]
                           else not o["has_audio"],
        "codec_h264_yuv420p": o["codec"] == "h264" and o["pix_fmt"] == "yuv420p",
        "size_ok": [o["width"], o["height"]] == [s["width"] * scale, s["height"] * scale],
    }
    return {
        "src": Path(src).name, "dst": Path(dst).name,
        "src_frames": sf, "out_frames": of,
        "src_duration": s["format_duration"], "out_duration": o["format_duration"],
        "duration_diff_sec": round(dur_diff, 4),
        "src_fps": s["fps"], "out_fps": o["fps"],
        "src_audio": s["audio_codec"], "out_audio": o["audio_codec"],
        "out_size": [o["width"], o["height"]], "out_bytes": o["size_bytes"],
        "checks": checks, "pass": all(checks.values()),
    }

print("helpers ready; ffmpeg flags:", PASSTHROUGH)

## 6. Test clips — คลิปทดสอบ
**EN:** Generates a 5 s 1280x720 30 fps H.264+AAC clip (`testsrc2` + `sine`) and a 360p downscale
(audio copied). To use your own clip set `USE_UPLOAD = True` (trimmed to first 5 s, re-encoded to
H.264+AAC, resolution kept; keep it <= 1280x720 on the free T4). Skippable.
**TH:** สร้างคลิป 720p 5 วินาที และ 360p; อยากใช้คลิปตัวเองให้ตั้ง `USE_UPLOAD = True`

In [ ]:
USE_UPLOAD = False      # True -> choose your own video in the upload dialog (cell skippable)
CLIP_SECONDS = 5

CLIP_MAIN = WORK / "clip_main.mp4"     # "720p" clip -> scale-2 job
CLIP_SMALL = WORK / "clip_small.mp4"   # 360p clip   -> scale-4 job

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    raw_in = WORK / ("user_" + Path(name).name)
    raw_in.write_bytes(uploaded[name])
    sh(["ffmpeg", "-v", "error", "-y", "-i", str(raw_in), "-t", str(CLIP_SECONDS),
        "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart", str(CLIP_MAIN)])
else:
    sh(["ffmpeg", "-v", "error", "-y",
        "-f", "lavfi", "-i", "testsrc2=size=1280x720:rate=30",
        "-f", "lavfi", "-i", "sine=frequency=440:sample_rate=48000",
        "-t", str(CLIP_SECONDS),
        "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart", str(CLIP_MAIN)])

sh(["ffmpeg", "-v", "error", "-y", "-i", str(CLIP_MAIN), "-vf", "scale=-2:360:flags=lanczos",
    "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p",
    "-c:a", "copy", "-movflags", "+faststart", str(CLIP_SMALL)])

for key, path in (("main", CLIP_MAIN), ("small", CLIP_SMALL)):
    info = probe(path)
    info["frames"] = count_frames(path)
    RESULTS["clips"][key] = info
    print(key, json.dumps(info))

## 7. VRAM probe: tile=0 vs tile=256 (and others) — วัด VRAM ตาม tile
**EN:** 2 real frames per setting, fp16, OOM is caught and recorded. `tile=0` means whole frame.
The first, unrecorded warm-up frame excludes CUDA/cuDNN init from timings.
**TH:** จับ OOM และบันทึกผล; เฟรมแรกเป็น warm-up ไม่นับเวลา

In [ ]:
PROBE_FRAMES = 2
PROBE_TILES = (128, 256, 512, 0)       # 0 (whole frame) last: if it OOMs nothing else is affected

t0 = time.perf_counter()
upsampler = build_upsampler("x4plus", tile=256)
RESULTS["model_load_sec"] = round(time.perf_counter() - t0, 2)
print("x4plus loaded in", RESULTS["model_load_sec"], "s")


def vram_probe(clip, outscale, label):
    frames = read_frames_bgr(clip, PROBE_FRAMES)
    upsampler.tile_size = 256
    enhance_frame(upsampler, frames[0], outscale)     # warm-up (not recorded)
    cuda_sync()
    for tile in PROBE_TILES:
        free_gpu()
        torch.cuda.reset_peak_memory_stats()
        rec = {"case": label, "in_size": [frames[0].shape[1], frames[0].shape[0]],
               "outscale": outscale, "tile": tile, "oom": False, "error": None}
        upsampler.tile_size = tile
        try:
            t = time.perf_counter()
            for f in frames:
                enhance_frame(upsampler, f, outscale)
            cuda_sync()
            rec["sec_per_frame"] = round((time.perf_counter() - t) / len(frames), 4)
        except Exception as e:  # noqa: BLE001
            rec["oom"] = is_oom(e)
            rec["error"] = f"{type(e).__name__}: {str(e)[:200]}"
        rec["peak_alloc_mib"] = round(torch.cuda.max_memory_allocated() / 2**20, 1)
        RESULTS["vram_probe"].append(rec)
        print(rec)
        free_gpu()


vram_probe(CLIP_SMALL, 4, "x4_from_360p")
vram_probe(CLIP_MAIN, 2, "x2_from_720p")

## 8. Enhance runs (full 5 s clips) — รันจริงทั้งคลิป
**EN:** (a) scale 4: `RealESRGAN_x4plus` on the 360p clip -> 1440p, libx264; (b) scale 2: x4plus with
`outscale=2` on the 720p clip -> 1440p, libx264; (c) repeat (a) with `h264_nvenc` if usable.
Edit `TILE_X4` / `TILE_X2` from the probe above if you want (default 256; 0 = no tiling).
Each job records per-stage wall time, s/frame and peak VRAM.
**TH:** (a) x4 จาก 360p (b) x2 จาก 720p (c) ซ้ำ (a) ด้วย NVENC ถ้าใช้ได้

In [ ]:
TILE_X4 = 256
TILE_X2 = 256

JOBS = [
    ("x4_360p_libx264", CLIP_SMALL, 4, "libx264", TILE_X4),
    ("x2_720p_libx264", CLIP_MAIN, 2, "libx264", TILE_X2),
]
if NVENC_USABLE:
    JOBS.append(("x4_360p_h264_nvenc", CLIP_SMALL, 4, "h264_nvenc", TILE_X4))
else:
    print("h264_nvenc not usable here -> skipping NVENC end-to-end run:", NVENC_NOTE)

OUT_DIR = WORK / "out"
OUT_DIR.mkdir(exist_ok=True)
JOB_FILES = {}
for label, clip, scale, encoder, tile in JOBS:
    dst = OUT_DIR / f"{label}.mp4"
    print(f"== {label}: tile={tile}, encoder={encoder}")
    st = enhance_video(clip, dst, upsampler, outscale=scale, encoder=encoder, tile=tile, label=label)
    RESULTS["jobs"].append(st)
    JOB_FILES[label] = (clip, dst, scale)
    print(json.dumps({k: st[k] for k in ("ok", "frames", "t_total_sec", "sec_per_frame_total",
                                          "sec_per_frame_model", "peak_alloc_mib", "error")}))

## 9. Encoder benchmark: libx264 vs h264_nvenc — เทียบ encoder แยกจากโมเดล
**EN:** Same 150 raw 2560x1440 frames (720p clip upscaled by ffmpeg, no AI) piped into each encoder,
so the number is pure encode throughput at the output resolution.
**TH:** วัดความเร็ว encode อย่างเดียวที่ความละเอียดผลลัพธ์ 2560x1440

In [ ]:
def encoder_bench(encoder, src, width=2560, height=1440, fps="30/1"):
    dst = WORK / f"bench_{encoder}.mp4"
    dec_log = open(WORK / "bench.decode.log", "wb")
    enc_log = open(WORK / f"bench_{encoder}.encode.log", "wb")
    dec_args = ["-vf", f"scale={width}:{height}:flags=bilinear"]
    n = 0
    t0 = time.perf_counter()
    dec = subprocess.Popen(decode_cmd(src, dec_args), stdout=subprocess.PIPE, stderr=dec_log)
    enc = subprocess.Popen(encode_cmd(dst, width, height, fps, encoder),
                           stdin=subprocess.PIPE, stderr=enc_log)
    try:
        fb = width * height * 3
        while len(buf := dec.stdout.read(fb)) == fb:
            enc.stdin.write(buf)
            n += 1
        enc.stdin.close()
        enc.wait()
        dec.wait()
        ok = enc.returncode == 0 and dec.returncode == 0
    finally:
        for p in (dec, enc):
            if p.poll() is None:
                p.kill()
        dec_log.close()
        enc_log.close()
    sec = time.perf_counter() - t0
    return {"encoder": encoder, "ok": ok, "frames": n, "size": [width, height],
            "t_total_sec": round(sec, 3), "encode_fps": round(n / sec, 2) if sec else None,
            "out_bytes": dst.stat().st_size if dst.exists() else None}


for enc_name in ["libx264"] + (["h264_nvenc"] if NVENC_USABLE else []):
    r = encoder_bench(enc_name, CLIP_MAIN)
    RESULTS["encoder_bench"].append(r)
    print(r)

## 10. Anime model smoke test — ทดสอบโมเดล anime
**EN:** `realesr-animevideov3` loads and runs 3 frames (360p, scale 4). Not a quality test.
**TH:** เช็กว่าโมเดล anime โหลดและรันได้เท่านั้น

In [ ]:
del upsampler
free_gpu()
anime = build_upsampler("anime", tile=256)
frames = read_frames_bgr(CLIP_SMALL, 3)
enhance_frame(anime, frames[0], 4)       # warm-up
cuda_sync()
free_gpu()
torch.cuda.reset_peak_memory_stats()
t = time.perf_counter()
for f in frames:
    out = enhance_frame(anime, f, 4)
cuda_sync()
RESULTS["anime_smoke"] = {
    "ok": True, "frames": len(frames), "out_size": [out.shape[1], out.shape[0]],
    "sec_per_frame": round((time.perf_counter() - t) / len(frames), 4),
    "peak_alloc_mib": round(torch.cuda.max_memory_allocated() / 2**20, 1),
}
print(RESULTS["anime_smoke"])
del anime
free_gpu()

## 11. Verify outputs — ตรวจผลลัพธ์
**EN:** For every successful job: `ffprobe -count_frames` equals input, duration diff <= 0.1 s,
same rational FPS, audio stream present (codec copied), H.264 yuv420p, size = input x scale.
**TH:** นับเฟรมจริงต้องเท่าต้นฉบับ, ระยะเวลาต่างไม่เกิน 0.1 วินาที, มีเสียง

In [ ]:
for label, (clip, dst, scale) in JOB_FILES.items():
    job = next(j for j in RESULTS["jobs"] if j["label"] == label)
    if not job["ok"]:
        RESULTS["verify"].append({"dst": dst.name, "pass": False, "note": "job failed: " + str(job["error"])})
        print(label, "-> job failed, nothing to verify")
        continue
    v = verify_output(clip, dst, scale)
    v["label"] = label
    RESULTS["verify"].append(v)
    print(label, "PASS" if v["pass"] else "FAIL", v["checks"])

## 12. Summary JSON — สรุปผล (copy this back / คัดลอกส่งกลับ)
**EN:** Copy everything between the lines. It contains no secrets (only versions, hashes, timings).
**TH:** คัดลอกทั้งก้อน JSON ส่งกลับ ไม่มีข้อมูลลับ

In [ ]:
RESULTS["all_verify_pass"] = bool(RESULTS["verify"]) and all(v["pass"] for v in RESULTS["verify"])
RESULTS["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
(WORK / "spike_results.json").write_text(json.dumps(RESULTS, indent=2, default=str))
print("=" * 20, "BEGIN SUMMARY JSON", "=" * 20)
print(json.dumps(RESULTS, indent=2, default=str))
print("=" * 20, "END SUMMARY JSON", "=" * 20)